# System Handler – AI Study Assistant  
## Notebook oficial: proyecto práctico de las primeras 2 semanas

Este notebook está pensado para que cualquier persona pueda construir un **chat educativo con IA** usando **OpenAI + Gradio + Jupyter Notebook**.

### Qué vas a construir
Un asistente tipo chat que:
- responde preguntas como tutor,
- explica con lenguaje claro,
- controla el contexto,
- muestra métricas básicas,
- y te deja experimentar con los parámetros del modelo.

### Cómo conecta con los primeros 10 reels

**Semana 1 – Fundamentos**
1. **¿Qué es OpenAI?** → Usamos la API de OpenAI desde Python.  
2. **¿Qué es realmente un LLM?** → El modelo genera texto prediciendo la siguiente parte más probable.  
3. **¿Qué es un Token?** → Veremos cuántos tokens usa cada respuesta.  
4. **¿Qué es el Contexto?** → Limitaremos el historial del chat para no crecer sin control.  
5. **¿Qué es un Prompt?** → Diseñaremos un prompt profesional para que el tutor responda mejor.

**Semana 2 – Control del modelo**
6. **Roles: system, user, assistant** → Construiremos mensajes con estos roles.  
7. **Temperature** → Controlaremos creatividad vs precisión.  
8. **top_p** → Limitaremos el rango de probabilidad del modelo.  
9. **max_output_tokens** → Limitaremos el tamaño de la respuesta.  
10. **Determinismo** → Activaremos un modo más estable para reducir variación.

### Requisitos
- Tener una API key de OpenAI
- Usar Jupyter Notebook, JupyterLab, VS Code o Google Colab
- Tener conexión a internet para que el chat funcione

## Celda 1 — Instalar dependencias

Esta celda sirve para instalar las librerías que usaremos, todo se instalara con uv:

- **uv**: uv (desarrollado por Astral) es un gestor de paquetes y entornos de Python extremadamente rápido y eficiente, escrito en Rust. Diseñado para reemplazar herramientas tradicionales como pip, pip-tools, virtualenv y poetry, ofrece una solución única, moderna y rápida para instalar y gestionar dependencias, optimizando drásticamente el flujo de trabajo de desarrollo.

ve a astral y sigue las instrucciones de instalacion, para cualquier error puedes consutlar cualquier gpt, como chatgpt, claude, entre otros.

https://docs.astral.sh/uv/getting-started/installation/


> Si ya las tienes instaladas, no pasa nada.

## Celda 2 — Importaciones

Aquí cargamos las librerías necesarias.

- `os` nos ayuda a leer la API key
- `textwrap` nos ayuda a dar formato
- `OpenAI` es el cliente para usar la API
- `gradio` crea la interfaz visual

In [1]:
import os
import textwrap
from typing import List, Dict, Any

import gradio as gr
from openai import OpenAI
from anthropic import Anthropic

## Celda 3 — Configura tus API keys

### API Keys

Para obtener una API key de **OpenAI**, debes haner iniciado sesión:

https://platform.openai.com/api-keys

Para **Anthropic** debes ir a:

https://platform.claude.com/settings/keys

### Archivo .env

En la raíz del proyecto crea un archivo llamado:

`.env`

Y dentro coloca algo como esto:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxx
ANTHROPIC_API_KEY=sk-ant-xxxxxxxxxxxx
```

### Provedores

Ahora el notebook puede trabajar con **dos proveedores**:

- **OpenAI** → por ejemplo, GPT / ChatGPT
- **Anthropic** → Claude

Puedes usar uno o los dos.

### Variables esperadas
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`

> Recomendación: para proyectos reales, usa variables de entorno o secretos, no texto plano.

In [ ]:
# Opción A: pega tus keys aquí entre comillas
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

if not openai_api_key and not anthropic_api_key:
    raise ValueError(
        "No encontré ninguna API key. Configura OPENAI_API_KEY o ANTHROPIC_API_KEY."
    )

print("Estado de credenciales:")
print(f"- OpenAI disponible: {bool(openai_api_key)}")
print(f"- Anthropic disponible: {bool(anthropic_api_key)}")

## Celda 4 — Crear los clientes y definir los modelos

Aquí hacemos la conexión con los proveedores disponibles.

Esto conecta con el **Reel 1**:
- una cosa es la empresa,
- otra el modelo,
- y otra tu aplicación.

En este notebook, tu app puede trabajar con más de un proveedor y dejar que el usuario elija.

In [ ]:
openai_client = OpenAI(api_key=openai_api_key) if openai_api_key else None
anthropic_client = Anthropic(api_key=anthropic_api_key) if anthropic_api_key else None

MODEL_CATALOG = {
    "OpenAI / ChatGPT": [
        "gpt-4.1-mini",
        "gpt-4.1",
        "gpt-5-mini"
    ],
    "Anthropic / Claude": [
        "claude-haiku-4-5",
        "claude-sonnet-4-5",
        "claude-opus-4-1"
    ],
}

AVAILABLE_PROVIDERS = [
    provider for provider in MODEL_CATALOG
    if (provider == "OpenAI / ChatGPT" and openai_client is not None)
    or (provider == "Anthropic / Claude" and anthropic_client is not None)
]

DEFAULT_PROVIDER = AVAILABLE_PROVIDERS[0]
DEFAULT_MODEL = MODEL_CATALOG[DEFAULT_PROVIDER][0]

print("Proveedores disponibles:", AVAILABLE_PROVIDERS)
print("Proveedor por defecto:", DEFAULT_PROVIDER)
print("Modelo por defecto:", DEFAULT_MODEL)

## Celda 5 — Diseñar el prompt del tutor

Esto conecta con el **Reel 5: Prompt**.

Un buen prompt no es “explícame algo”.
Un buen prompt define:

- quién es el asistente,
- cómo debe responder,
- para quién habla,
- y qué formato debe seguir.

Aquí vamos a crear un tutor claro, amable y útil.

In [ ]:
SYSTEM_PROMPT = """
Eres System Handler AI Study Assistant, un tutor experto, claro y paciente.

Tu misión es explicar cualquier tema de forma fácil de entender para cualquier persona,
incluyendo principiantes. Evita tecnicismos innecesarios.

Reglas de respuesta:
1. Explica primero la idea principal en palabras simples.
2. Después da un ejemplo real o cotidiano.
3. Si ayuda, usa una analogía sencilla.
4. Cierra con un error común o una recomendación práctica.
5. Si el usuario pregunta por código, explica el código antes de mostrarlo.
6. Si el usuario parece principiante, usa tono didáctico y ordenado.
7. No inventes datos. Si no sabes algo, dilo con honestidad.

Formato preferido:
- Idea principal
- Ejemplo
- Analogía o tip práctico
- Error común o siguiente paso
"""

print(SYSTEM_PROMPT)

## Celda 6 — Parámetros por defecto del modelo

Esto conecta con varios reels:

- **Temperature** → creatividad vs precisión
- **top_p** → rango de probabilidad
- **max_output_tokens** → tamaño máximo de la respuesta
- **determinismo** → modo más estable

### Regla fácil de recordar
- Menor `temperature` = respuestas más estables
- Menor `top_p` = menos opciones posibles
- Menor `max_output_tokens` = respuestas más cortas

In [ ]:
DEFAULT_TEMPERATURE = 0.3
DEFAULT_TOP_P = 0.9
DEFAULT_MAX_OUTPUT_TOKENS = 300
MAX_HISTORY_TURNS = 6  # cada turno es una pregunta + una respuesta

print("Parámetros base listos.")

## Celda 7 — Función para recortar el contexto

Esto conecta con el **Reel 4: Contexto**.

Un modelo no recuerda infinito.
Si sigues agregando mensajes para siempre:
- sube el costo,
- crece el uso de tokens,
- y el sistema se vuelve menos eficiente.

Por eso vamos a limitar el historial.

In [6]:
def trim_history(history, max_turns=6):
    max_messages = max_turns * 2
    return (history or [])[-max_messages:]

## Celda 8 — Construir mensajes con roles

Esto conecta con el **Reel 6: Roles**.

Cada mensaje tendrá uno de estos roles:

- `system` → reglas del tutor
- `user` → lo que pregunta la persona
- `assistant` → lo que responde la IA

Eso le da estructura profesional a la conversación. Ejemplo:

```
system: instrucciones del tutor
user: primera pregunta
assistant: primera respuesta
user: nueva pregunta
```

In [7]:
def build_messages(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for item in trim_history(history, MAX_HISTORY_TURNS):
        if not isinstance(item, dict):
            continue

        role = item.get("role")
        content = item.get("content")

        if role in {"user", "assistant"} and isinstance(content, str):
            messages.append({
                "role": role,
                "content": content
            })

    messages.append({"role": "user", "content": message})
    return messages

## Celda 9 — Llamar al modelo y obtener respuesta

Aquí ocurre la parte más importante del proyecto.

Esto conecta con:
- **Reel 2** → el LLM genera texto
- **Reel 3** → vemos uso de tokens
- **Reel 7** → `temperature`
- **Reel 8** → `top_p`
- **Reel 9** → `max_output_tokens`
- **Reel 10** → modo más estable

### Idea importante
Ahora el notebook puede llamar a **OpenAI** o a **Claude** según lo que elijas en la interfaz.

### Nota práctica
Para mantener compatibilidad simple:
- con **OpenAI** enviamos `temperature` y `top_p`
- con **Claude** enviamos `temperature` y omitimos `top_p` en la llamada

In [8]:
def extract_claude_text(response) -> str:
    parts = []
    for block in getattr(response, "content", []):
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    return "\n".join(parts).strip()


def normalize_provider(provider_name: str) -> str:
    value = (provider_name or "").strip().lower()

    if value in {"openai", "openai / chatgpt", "chatgpt", "gpt"}:
        return "OpenAI / ChatGPT"

    if value in {"anthropic", "anthropic / claude", "claude"}:
        return "Anthropic / Claude"

    raise ValueError(f"Proveedor no soportado: {provider_name}")


def ask_openai(
    user_message: str,
    history: List[Dict[str, str]],
    model_name: str,
    temperature: float,
    top_p: float,
    max_output_tokens: int,
    stable_mode: bool
) -> Dict[str, Any]:
    if openai_client is None:
        raise ValueError("OPENAI_API_KEY no está configurada.")

    if stable_mode:
        temperature = min(temperature, 0.2)
        top_p = min(top_p, 0.5)

    messages = build_messages(user_message, history)

    response = openai_client.responses.create(
        model=model_name,
        input=messages,
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens
    )

    usage = getattr(response, "usage", None)

    return {
        "answer": response.output_text,
        "provider": "OpenAI / ChatGPT",
        "model_name": model_name,
        "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
        "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
        "total_tokens": getattr(usage, "total_tokens", None) if usage else None,
        "temperature_used": temperature,
        "top_p_used": top_p,
        "max_output_tokens_used": max_output_tokens,
        "messages_sent": len(messages),
    }


def ask_claude(
    user_message: str,
    history: List[Dict[str, str]],
    model_name: str,
    temperature: float,
    top_p: float,
    max_output_tokens: int,
    stable_mode: bool
) -> Dict[str, Any]:
    if anthropic_client is None:
        raise ValueError("ANTHROPIC_API_KEY no está configurada.")

    if stable_mode:
        temperature = min(temperature, 0.2)

    messages = build_messages(user_message, history)
    claude_messages = [m for m in messages if m["role"] != "system"]

    response = anthropic_client.messages.create(
        model=model_name,
        system=SYSTEM_PROMPT,
        messages=claude_messages,
        temperature=temperature,
        max_tokens=max_output_tokens
    )

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) if usage else None
    output_tokens = getattr(usage, "output_tokens", None) if usage else None
    total_tokens = (input_tokens or 0) + (output_tokens or 0) if usage else None

    return {
        "answer": extract_claude_text(response),
        "provider": "Anthropic / Claude",
        "model_name": model_name,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "temperature_used": temperature,
        "top_p_used": "No enviado a Claude en este notebook",
        "max_output_tokens_used": max_output_tokens,
        "messages_sent": len(messages),
    }


def ask_model(
    provider,
    model_name,
    user_message,
    history,
    temperature,
    top_p,
    max_output_tokens,
    stable_mode
):
    normalized_provider = normalize_provider(provider)

    if normalized_provider == "OpenAI / ChatGPT":
        return ask_openai(
            user_message=user_message,
            history=history,
            model_name=model_name,
            temperature=temperature,
            top_p=top_p,
            max_output_tokens=max_output_tokens,
            stable_mode=stable_mode
        )

    if normalized_provider == "Anthropic / Claude":
        return ask_claude(
            user_message=user_message,
            history=history,
            model_name=model_name,
            temperature=temperature,
            top_p=top_p,
            max_output_tokens=max_output_tokens,
            stable_mode=stable_mode
        )

    raise ValueError(f"Proveedor no soportado: {provider}")

## Celda 10 — Probar el modelo sin interfaz

Antes de hacer el chat bonito, probamos con una pregunta simple.

Esto ayuda a verificar que:
- la API key funciona,
- el modelo responde,
- y el prompt está bien diseñado.

In [ ]:
demo_history = []

test = ask_model(
    provider=DEFAULT_PROVIDER,
    model_name=DEFAULT_MODEL,
    user_message="Explícame qué es un token de forma muy sencilla.",
    history=demo_history,
    temperature=DEFAULT_TEMPERATURE,
    top_p=DEFAULT_TOP_P,
    max_output_tokens=DEFAULT_MAX_OUTPUT_TOKENS,
    stable_mode=True
)

print(test["answer"])
print("\nMétricas:")
print(test)

## Celda 11 — Crear la lógica del chat para Gradio

Ahora conectamos la función del modelo con la interfaz visual.

La idea es:
1. el usuario elige proveedor y modelo,
2. escribe su pregunta,
3. se llama al proveedor correcto,
4. se actualiza el historial,
5. y mostramos métricas de la última respuesta.

In [10]:
def update_model_choices(provider_name: str):
    normalized_provider = normalize_provider(provider_name)
    models = MODEL_CATALOG[normalized_provider]
    return gr.update(choices=models, value=models[0])


def chat_fn(message, history, provider, model_name, temperature, top_p, max_output_tokens, stable_mode):
    history = history or []

    result = ask_model(
        provider=provider,
        model_name=model_name,
        user_message=message,
        history=history,
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens,
        stable_mode=stable_mode
    )

    assistant_text = result["answer"]

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": assistant_text}
    ]

    metrics_text = f"""
### Métricas de la última respuesta
- **Proveedor:** {result['provider']}
- **Modelo:** {result['model_name']}
- **Mensajes enviados al modelo:** {result['messages_sent']}
- **Input tokens:** {result.get('input_tokens')}
- **Output tokens:** {result.get('output_tokens')}
- **Total tokens:** {result.get('total_tokens')}
- **Temperature usada:** {result['temperature_used']}
- **top_p usado:** {result.get('top_p_used')}
- **Max output tokens:** {result['max_output_tokens_used']}
"""

    return history, metrics_text

## Celda 12 — Crear la interfaz visual con Gradio

Aquí aparece el chatbot visual.

Ahora tendrá:
- selector de **proveedor**
- selector de **modelo**
- sliders para parámetros
- switch de modo estable
- panel de métricas
- botón para limpiar conversación

In [11]:
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 🤖 System Handler – AI Study Assistant
        Aprende cualquier tema con un tutor de IA claro, estructurado y fácil de entender.

        **Ideas que estás aplicando aquí:**
        OpenAI, Claude, LLM, tokens, contexto, prompt, roles, temperature, top_p, max_output_tokens y modo estable.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Chat",
                height=500
            )   

            user_input = gr.Textbox(
                label="Escribe tu pregunta",
                placeholder="Ejemplo: Explícame qué es un microservicio como si tuviera 12 años.",
                lines=3
            )

            with gr.Row():
                send_btn = gr.Button("Enviar")
                clear_btn = gr.Button("Limpiar chat")

        with gr.Column(scale=2):
            provider_dropdown = gr.Dropdown(
                choices=AVAILABLE_PROVIDERS,
                value=DEFAULT_PROVIDER,
                label="Proveedor"
            )

            model_dropdown = gr.Dropdown(
                choices=MODEL_CATALOG[DEFAULT_PROVIDER],
                value=DEFAULT_MODEL,
                label="Modelo"
            )

            temperature_slider = gr.Slider(
                minimum=0.0,
                maximum=1.5,
                value=DEFAULT_TEMPERATURE,
                step=0.1,
                label="Temperature"
            )

            top_p_slider = gr.Slider(
                minimum=0.1,
                maximum=1.0,
                value=DEFAULT_TOP_P,
                step=0.1,
                label="top_p"
            )

            max_tokens_slider = gr.Slider(
                minimum=50,
                maximum=800,
                value=DEFAULT_MAX_OUTPUT_TOKENS,
                step=10,
                label="Max output tokens"
            )

            stable_mode_checkbox = gr.Checkbox(
                value=True,
                label="Modo estable (más consistencia, menos creatividad)"
            )

            metrics_markdown = gr.Markdown(
                """
                ### Métricas
                Aquí aparecerán las métricas de la última respuesta.
                """
            )

    provider_dropdown.change(
        fn=update_model_choices,
        inputs=[provider_dropdown],
        outputs=[model_dropdown]
    )

    send_btn.click(
        fn=chat_fn,
        inputs=[
            user_input,
            chatbot,
            provider_dropdown,
            model_dropdown,
            temperature_slider,
            top_p_slider,
            max_tokens_slider,
            stable_mode_checkbox
        ],
        outputs=[chatbot, metrics_markdown]
    ).then(
        lambda: "",
        outputs=[user_input]
    )


    user_input.submit(
        fn=chat_fn,
        inputs=[
            user_input,
            chatbot,
            provider_dropdown,
            model_dropdown,
            temperature_slider,
            top_p_slider,
            max_tokens_slider,
            stable_mode_checkbox
        ],
        outputs=[chatbot, metrics_markdown]
    ).then(
        lambda: "",
        outputs=[user_input]
    )

    clear_btn.click(
        lambda: ([], "### Métricas\nChat reiniciado."),
        outputs=[chatbot, metrics_markdown]
    )


## Celda 13 — Lanzar el chat

Ejecuta esta celda para abrir la interfaz.

### Si estás en:
- **Jupyter local / VS Code** → normalmente se abrirá debajo o en un enlace local
- **Google Colab** → puede abrirse en un enlace temporal

In [ ]:
demo.launch(
    share=False,
    inline=True
)

## Celda 14 — Guía fácil para experimentar

Prueba estas combinaciones para ver cómo cambia el comportamiento del modelo.

### 1. Modo preciso
- Temperature: `0.2`
- top_p: `0.5`
- Max output tokens: `200`
- Modo estable: activado

Úsalo para:
- estudiar
- explicar conceptos
- respuestas más consistentes

### 2. Modo balanceado
- Temperature: `0.5`
- top_p: `0.8`
- Max output tokens: `300`
- Modo estable: desactivado

Úsalo para:
- aprender
- explorar ideas
- respuestas claras con algo de flexibilidad

### 3. Modo creativo
- Temperature: `0.9`
- top_p: `1.0`
- Max output tokens: `400`
- Modo estable: desactivado

Úsalo para:
- nombres
- slogans
- ideas creativas
- analogías distintas

### 4. Comparación entre proveedores
Haz la misma pregunta en:
- **OpenAI / ChatGPT**
- **Anthropic / Claude**

Y compara:
- estilo de explicación
- longitud
- claridad
- costo aproximado por tokens

## Celda 15 — Ejemplos de preguntas para probar

Aquí tienes prompts listos para copiar y pegar.

### Explicación simple
- Explícame qué es una API de forma muy fácil.
- Explícame qué es un token como si fuera dinero del taxímetro.
- Explícame qué es el contexto en un LLM con una analogía cotidiana.

### Programación
- Explícame qué hace un loop en Python con un ejemplo básico.
- Explícame la diferencia entre `if`, `elif` y `else`.
- Muéstrame un ejemplo simple de función en Python y explícalo línea por línea.

### Sistemas
- Explícame qué es un microservicio con un ejemplo de negocio.
- ¿Qué problema resuelve una cola como SQS?
- ¿Qué diferencia hay entre frontend y backend?

### Entrevistas
- Simula una pregunta de entrevista sobre Java y luego corrígeme mi respuesta.
- Hazme un quiz corto de Python básico.
- Dame 3 preguntas técnicas sobre APIs REST con respuestas simples.

## Celda 16 — Qué aprendiste realmente con este proyecto

Si entendiste este notebook, ya aplicaste los 10 temas y además diste un paso extra:
**comparar distintos proveedores en una misma interfaz.**

### Semana 1
1. **OpenAI** → usaste la empresa y su API  
2. **LLM** → viste cómo el modelo genera respuestas  
3. **Tokens** → mediste uso real  
4. **Contexto** → limitaste historial  
5. **Prompt** → diseñaste instrucciones profesionales  

### Semana 2
6. **Roles** → `system`, `user`, `assistant`  
7. **Temperature** → controlaste creatividad  
8. **top_p** → entendiste cómo influye la selección probable  
9. **max_output_tokens** → limitaste longitud  
10. **Determinismo** → activaste un modo más estable  

Y además:
- elegiste proveedor,
- elegiste modelo,
- y comparaste respuestas sobre el mismo problema.

En otras palabras:
**ya no estás usando IA como caja negra.**
La estás **probando, controlando y comparando**.

## Celda 17 — Próximos pasos recomendados

Si quieres convertir este notebook en un proyecto más serio, el siguiente paso natural es agregar:

1. **Memoria limitada por tema**  
2. **Carga de PDF o apuntes**  
3. **Modo examen / quiz**  
4. **Modo corrector de código**  
5. **Guardado de conversaciones**  
6. **Versión web fuera de Jupyter**

Ese camino lo convierte en un producto educativo real.